# 학습 데이터 전처리

불량을 1로 두고 분할한 뒤, 학습 데이터에서만 전처리 기준을 계산한다.

In [1]:
from pathlib import Path
import os
import sys
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from analysis import load_data, make_xy, split_data, ProcessFeatures, save_eda, run_experiment

data_path = Path(os.environ.get("MANUFACTURING_DATA", ROOT / "data" / "labeled_data.csv"))
data = load_data(data_path)
output = ROOT / "results" / "current"


In [2]:
X, y = make_xy(data)
X_train, X_test, y_train, y_test = split_data(X, y, seed=42)
display(pd.Series({'train':len(X_train), 'test':len(X_test), 'defects_train':int(y_train.sum()), 'defects_test':int(y_test.sum())}))
assert 'Reason' not in X and 'PART_FACT_SERIAL' not in X

train            6396
test             1600
defects_train      57
defects_test       14
dtype: int64

In [3]:
features = ProcessFeatures(scaling='standard', seed=42)
prepared_train = features.fit_transform(X_train)
prepared_test = features.transform(X_test)
bounds = [dict(product=key[0], equipment=key[1], lower=value[0]['Injection_Time'], upper=value[1]['Injection_Time'])
          for key, value in features.group_bounds_.items()]
display(pd.DataFrame(bounds))
display(prepared_train.head())


,product,equipment,lower,upper
0,CN7 W/S SIDE MLD'G RH,S14,9.424999,9.705001
1,CN7 W/S SIDE MLD'G LH,S14,9.424999,9.705001
2,"RG3 MOLD'G W/SHLD, RH",S14,-inf,inf
3,"RG3 MOLD'G W/SHLD, LH",S14,-inf,inf


,Injection_Time,Filling_Time,Plasticizing_Time,Cycle_Time,Clamp_Close_Time,Cushion_Position,Switch_Over_Position,Plasticizing_Position,Clamp_Open_Position,Max_Injection_Speed,...,Barrel_Temperature_6,Barrel_Temperature_7,Hopper_Temperature,Mold_Temperature_3,Mold_Temperature_4,Injection_Time_log,Filling_Time_sqrt,PART_NAME,EQUIP_CD,EQUIP_NAME
236,0.441240,0.450721,0.495399,-0.364437,0.411482,0.003637,-0.017954,0.515995,0.4409,-0.437522,...,-0.344666,-0.017686,-0.063758,2.394664,2.886549,0.438074,0.446387,CN7 W/S SIDE MLD'G RH,S14,650톤-우진2호기
7634,0.444453,0.458509,0.302108,-0.432225,0.338104,0.005399,-0.017954,0.422229,0.4409,-0.456335,...,-0.344666,-0.017686,-0.568468,-0.123231,0.126994,0.439654,0.452053,CN7 W/S SIDE MLD'G LH,S14,650톤-우진2호기
5659,0.444453,0.466297,0.474689,-0.387033,0.411482,0.006279,-0.017954,0.468193,0.4409,-0.437522,...,-0.344666,-0.017686,0.091535,0.296418,0.126994,0.439654,0.457712,CN7 W/S SIDE MLD'G RH,S14,650톤-우진2호기
3260,0.434813,0.435145,0.488496,-0.432225,0.411482,0.005399,-0.017954,0.453485,0.4409,-0.441284,...,-0.293797,-0.017686,0.673891,0.632137,0.490094,0.434910,0.435037,CN7 W/S SIDE MLD'G LH,S14,650톤-우진2호기
1108,0.425173,0.419569,0.598948,-0.432225,0.411482,0.003637,-0.017954,0.451646,0.4409,-0.411183,...,-0.446419,-0.017686,1.450367,1.387506,0.998432,0.430153,0.423662,CN7 W/S SIDE MLD'G LH,S14,650톤-우진2호기


In [4]:
from analysis import make_pipeline
from sklearn.linear_model import LogisticRegression
pipeline = make_pipeline('standard', LogisticRegression(max_iter=2000, solver='liblinear', random_state=42))
pipeline.fit(X_train, y_train)
encoded_test = pipeline.named_steps['encoding'].transform(pipeline.named_steps['features'].transform(X_test))
display(encoded_test.shape)

(1600, 40)

제품·설비별 IQR clipping과 log·sqrt 변환은 ProcessFeatures에, 범주 인코딩과 SMOTENC는 Pipeline에 들어 있다. 교차검증에서는 각 fold마다 다시 맞춘다.